In [1]:
import planetary_computer as pc
from pystac_client import Client
import pandas as pd
import numpy as np
import geopy.distance as distance
from datetime import timedelta
from time import time

import rioxarray
from IPython.display import Image
from PIL import Image as PILImage

In [3]:

def get_bounding_box(latitude, longitude, meter_buffer=50000):
    """
    Given a latitude, longitude, and buffer in meters, returns a bounding
    box around the point with the buffer on the left, right, top, and bottom.

    Returns a list of [minx, miny, maxx, maxy]
    """
    distance_search = distance.distance(meters=meter_buffer)

    # calculate the lat/long bounds based on ground distance
    # bearings are cardinal directions to move (south, west, north, and east)
    min_lat = distance_search.destination((latitude, longitude), bearing=180)[0]
    min_long = distance_search.destination((latitude, longitude), bearing=270)[1]
    max_lat = distance_search.destination((latitude, longitude), bearing=0)[0]
    max_long = distance_search.destination((latitude, longitude), bearing=90)[1]

    return [min_long, min_lat, max_long, max_lat]



In [4]:

def get_date_range(date, time_buffer_days=15):
    """Get a date range to search for in the planetary computer based
    on a sample's date. The time range will include the sample date
    and time_buffer_days days prior

    Returns a string"""
    datetime_format = "%Y-%m-%d"
    range_start = pd.to_datetime(date) - timedelta(days=time_buffer_days)
    range_end = pd.to_datetime(date) + timedelta(days=time_buffer_days)
    date_range = f"{range_start.strftime(datetime_format)}/{range_end.strftime(datetime_format)}"

    return date_range



In [5]:
catalog = Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1", modifier=pc.sign_inplace
)


In [7]:
df = pd.read_excel(r"C:\Users\KostasPikounis\OneDrive_Inlecom_Personal\OneDrive - INLECOM\Amfitrite\task2\IWD\tick tick bloom data\junk4.xlsx")
d1 = df[df.case == 449]  #35
test_dates = d1[d1.date == "2018-05-21"]
print(test_dates.shape)
p_lon = np.mean([min(test_dates.lon),max(test_dates.lon)])
p_lat = np.mean([min(test_dates.lat),max(test_dates.lat)])
print(p_lon, p_lat, set(test_dates.date))


(19, 17)
-79.13141456 35.723314869999996 {Timestamp('2018-05-21 00:00:00')}


In [59]:
test_dates

,uid,data_provider,region,lat,lon,date,time,year,month,day,abun,severity,distance_to_water_m,cyfi class,case,case_str,cluster_size
554,anrk,NC_Division_of_Water_Resources_NC_Department_o...,south,35.722290,-79.133318,2018-05-21,0:01:00,2018,5,21,0.145,1,1010.0,Low,449,449,117
567,dihj,NC_Division_of_Water_Resources_NC_Department_o...,south,35.724340,-79.129511,2018-05-21,0:01:00,2018,5,21,31.540,2,811.0,Moderate,449,449,117
571,dxqy,NC_Division_of_Water_Resources_NC_Department_o...,south,35.722264,-79.133367,2018-05-21,0:01:00,2018,5,21,0.290,1,1018.0,Low,449,449,117
574,edmh,NC_Division_of_Water_Resources_NC_Department_o...,south,35.722317,-79.133268,2018-05-21,0:01:00,2018,5,21,0.145,1,1010.0,Low,449,449,117
575,eetb,NC_Division_of_Water_Resources_NC_Department_o...,south,35.723275,-79.131489,2018-05-21,0:01:00,2018,5,21,3.389,1,885.0,Low,449,449,117
580,flph,NC_Division_of_Water_Resources_NC_Department_o...,south,35.724393,-79.129412,2018-05-21,0:01:00,2018,5,21,4.696,1,800.0,Low,449,449,117
584,guif,NC_Division_of_Water_Resources_NC_Department_o...,south,35.724419,-79.129363,2018-05-21,0:01:00,2018,5,21,60.420,2,807.0,Moderate,449,449,117
589,hknm,NC_Division_of_Water_Resources_NC_Department_o...,south,35.722210,-79.133466,2018-05-21,0:01:00,2018,5,21,1.017,1,1023.0,Low,449,449,117
590,htoo,NC_Division_of_Water_Resources_NC_Department_o...,south,35.723408,-79.131242,2018-05-21,0:01:00,2018,5,21,2.582,1,868.0,Low,449,449,117
596,jekg,NC_Division_of_Water_Resources_NC_Department_o...,south,35.724286,-79.129610,2018-05-21,0:01:00,2018,5,21,61.046,2,810.0,Moderate,449,449,117


In [8]:

bbox = get_bounding_box(p_lat, p_lon, meter_buffer=3000)
print(bbox)


[-79.16457194252429, 35.69627654781201, -79.0982571774757, 35.75035307044336]


In [9]:

date_range = get_date_range(test_dates.date.iloc[0])
date_range


'2018-05-06/2018-06-05'

In [10]:

search = catalog.search(
    collections=["sentinel-2-l2a"], bbox=bbox, datetime=date_range
)


In [11]:

t1 = time()
items = [item for item in search.get_all_items()]
t2 = time()
len(items)



C:\Users\KostasPikounis\anaconda3\envs\AMFITRITE\lib\site-packages\pystac_client\item_search.py:940: FutureWarning: get_all_items() is deprecated, use item_collection() instead.
  warnings.warn(


6

In [12]:

item_details = pd.DataFrame(
    [
        {
            "datetime": item.datetime.strftime("%Y-%m-%d"),
            "platform": item.properties["platform"],
            "min_long": item.bbox[0],
            "max_long": item.bbox[2],
            "min_lat": item.bbox[1],
            "max_lat": item.bbox[3],
            "bbox": item.bbox,
            "item_obj": item,
        }
        for item in items
    ]
)

# check which rows actually contain the sample location
item_details["contains_sample_point"] = (
    (item_details.min_lat < p_lat)
    & (item_details.max_lat > p_lat)
    & (item_details.min_long < p_lon)
    & (item_details.max_long > p_lon)
)

print(
    f"Filtering from {len(item_details)} returned to {item_details.contains_sample_point.sum()} items that contain the sample location"
)

item_details = item_details[item_details["contains_sample_point"]]
item_details[["datetime", "platform", "contains_sample_point", "bbox"]].sort_values(
    by="datetime"
)



Filtering from 6 returned to 6 items that contain the sample location


,datetime,platform,contains_sample_point,bbox
5,2018-05-09,Sentinel-2B,True,"[-79.9021462, 35.1330063, -78.6687854, 36.1397..."
4,2018-05-14,Sentinel-2A,True,"[-79.9021462, 35.1330063, -78.6687854, 36.1397..."
3,2018-05-19,Sentinel-2B,True,"[-79.90216, 35.1330063, -78.66879, 36.1397407]"
2,2018-05-24,Sentinel-2A,True,"[-79.90216, 35.1330063, -78.66879, 36.1397407]"
1,2018-05-29,Sentinel-2B,True,"[-79.90216, 35.1330063, -78.66879, 36.1397407]"
0,2018-06-03,Sentinel-2A,True,"[-79.90216, 35.1330063, -78.66879, 36.1397407]"


In [14]:

sel1 = item_details[item_details.contains_sample_point == True]

sel1['date_difference'] = (pd.to_datetime(sel1['datetime']) - test_dates.date.iloc[0]).dt.days
min_diff_index = sel1['date_difference'].abs().idxmin()
best_row = sel1.loc[[min_diff_index]]

item = best_row.item_obj.iloc[0]

item

<Item id=S2B_MSIL2A_20180519T155819_R097_T17SPV_20201012T115343>

In [15]:
for asset_key, asset in item.assets.items():
    print(f"{asset_key:<25} - {asset.title}")

AOT                       - Aerosol optical thickness (AOT)
B01                       - Band 1 - Coastal aerosol - 60m
B02                       - Band 2 - Blue - 10m
B03                       - Band 3 - Green - 10m
B04                       - Band 4 - Red - 10m
B05                       - Band 5 - Vegetation red edge 1 - 20m
B06                       - Band 6 - Vegetation red edge 2 - 20m
B07                       - Band 7 - Vegetation red edge 3 - 20m
B08                       - Band 8 - NIR - 10m
B09                       - Band 9 - Water vapor - 60m
B11                       - Band 11 - SWIR (1.6) - 20m
B12                       - Band 12 - SWIR (2.2) - 20m
B8A                       - Band 8A - Vegetation red edge 4 - 20m
SCL                       - Scene classfication map (SCL)
WVP                       - Water vapour (WVP)
visual                    - True color image
preview                   - Thumbnail
safe-manifest             - SAFE manifest
granule-metadata          - Granul

In [16]:
# see the whole image
img = Image(url=item.assets["rendered_preview"].href, width=500)

Image(url=item.assets["rendered_preview"].href, width=500)

In [18]:
test = rioxarray.open_rasterio(pc.sign(item.assets["visual"].href))

In [21]:
test2 = test.to_numpy()

In [41]:
test2[2]

array([[255, 255, 255, ..., 255, 255, 255],
       [255, 255, 255, ..., 255, 255, 255],
       [255, 255, 255, ..., 255, 255, 255],
       ...,
       [255, 255, 255, ..., 255, 255, 255],
       [255, 255, 255, ..., 255, 255, 255],
       [255, 255, 255, ..., 255, 255, 255]],
      shape=(10980, 10980), dtype=uint8)

In [42]:
mask = test2 != 255

In [44]:
indices_tuple = np.nonzero(mask)

In [45]:
indices_tuple

(array([0, 0, 0, ..., 2, 2, 2], shape=(18823871,)),
 array([    0,     0,     0, ..., 10979, 10979, 10979], shape=(18823871,)),
 array([ 555,  556,  557, ..., 1542, 1543, 1544], shape=(18823871,)))

In [46]:
test2[0][0][556]

np.uint8(240)

In [49]:
scl = rioxarray.open_rasterio(pc.sign(item.assets["SCL"].href))

In [50]:
scl2 = scl.to_numpy()

In [54]:
scl2

array([[[9, 8, 9, ..., 8, 8, 8],
        [9, 8, 9, ..., 8, 8, 8],
        [8, 9, 8, ..., 8, 8, 8],
        ...,
        [9, 9, 8, ..., 8, 8, 8],
        [9, 9, 8, ..., 8, 8, 8],
        [9, 9, 9, ..., 8, 8, 8]]], shape=(1, 5490, 5490), dtype=uint8)

In [58]:
unique_values, counts = np.unique(scl2, return_counts=True)
print(unique_values)
print(counts)
print(sum(counts), 5490*5490)
for i, j in zip(unique_values, counts):
    print(i, j, round(100*j/(5490*5490),2))

[ 2  3  4  5  6  7  8  9 10 11]
[     209      333   111115     3532       58   236215 22031045  7305183
   164614   287796]
30140100 30140100
2 209 0.0
3 333 0.0
4 111115 0.37
5 3532 0.01
6 58 0.0
7 236215 0.78
8 22031045 73.1
9 7305183 24.24
10 164614 0.55
11 287796 0.95
